# 50-chart live demo — Kaggle + Qwen 2.5 14B

Этот notebook запускает **живой** агент с Qwen 2.5 14B (Q4_K_M, 3 шарда) на
Kaggle GPU и строит demo-набор **10 станций × 14–18 мая 2024 = 50 HTML**.

Это не 300-chart gate. 50-графиковый набор нужен для показа: он быстрее,
покрывает все 10 станций и проверяет matrix → project → export → ZIP.

## Настройки Kaggle

1. **Accelerator:** GPU T4 x2 (или P100; 16 GB).
2. **Internet:** ON — нужен GitHub, HuggingFace, INTERMAGNET, OMNI, Kyoto.
3. **Persistence:** Files (чтобы `models/` и `exports/` пережили сессию).
4. **Add-ons → Secrets:** `SUPERMAG_LOGON` не обязателен для 50-chart demo.
   Если хотите шести-источниковый контекст, добавьте секрет с логином SuperMAG.

## Время

- установка wheel: 1–2 мин;
- скачивание модели: 5–12 мин;
- 50-графиковая матрица: ~10–20 мин;
- итого: ~20–35 мин.

## Ожидаемый результат

- `projects/demo_storm_may2024/` — 50 HTML, `charts.json`, CSV, manifest;
- `exports/demo_storm_may2024.zip` — 50 графиков;
- `exports/storm_report_2024_demo.zip` — custom-вид с папками `1/2/3`;
- ABG 13 May — ожидаемый `no_data`;
- multi-turn: H, затем D из того же handle.

> Важно: 14B может ошибаться в планировании. Notebook печатает все tool calls и
> не скрывает partial/empty. Если matrix не даст 50 графиков, ниже есть
> детерминированный fallback `demo_make_50.py`, но его нужно явно запускать
> отдельно и не выдавать за live-результат модели.


In [ ]:
# ── 1. Репозиторий: свежий GitHub checkout ──────────────────────────────
import os, sys, subprocess, shutil
from pathlib import Path

REPO = "https://github.com/IgorCim/geomagnetic-agent.git"
DIR = Path("/kaggle/working/geomagnetic-agent")

if (DIR / ".git").is_dir():
    r = subprocess.run(
        ["git", "-C", str(DIR), "pull", "--ff-only"],
        capture_output=True, text=True,
    )
    print(r.stdout.strip())
    print(r.stderr.strip())
    if r.returncode != 0:
        raise SystemExit(
            "git pull не удался. Проверьте Internet и отсутствие локальных "
            "изменений в /kaggle/working/geomagnetic-agent."
        )
else:
    if DIR.exists():
        shutil.rmtree(DIR)
    subprocess.run(["git", "clone", REPO, str(DIR)], check=True)

shutil.rmtree(DIR / "__pycache__", ignore_errors=True)
os.chdir(DIR)
if str(DIR) not in sys.path:
    sys.path.insert(0, str(DIR))

import agent_core as ac
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=DIR, text=True
).strip()
print("cwd       :", Path.cwd())
print("commit    :", commit)
print("revision  :", ac.code_revision())
assert len(ac.SYSTEM_PROMPT) <= 3200
budget = ac.prompt_budget(ac.SYSTEM_PROMPT, ac.TOOL_SCHEMAS, ac.N_CTX)
print("prompt    :", len(ac.SYSTEM_PROMPT), "chars; est_tokens", budget["est_tokens"])


In [ ]:
# ── 2. CUDA-wheel llama.cpp + зависимости ────────────────────────────────
# На PyPI обычно только sdist; сборка CUDA занимает 20–40 минут. Ставим
# готовый wheel. Если Kaggle обновил образ и wheel не подошёл — смотрите
# ошибку ниже; не заменяйте его CPU-сборкой для live demo.
!pip install -q --force-reinstall --no-deps https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35-cu121/llama_cpp_python-0.3.35-py3-none-manylinux_2_35_x86_64.whl
!pip install -q "jinja2>=3.1.2" "diskcache>=5.6" pandas plotly requests huggingface_hub pyarrow gradio

import llama_cpp
print("llama-cpp-python:", llama_cpp.__version__)


In [ ]:
# ── 3. GPU: torch + llama.cpp CUDA offload ───────────────────────────────
import torch
from llama_cpp import llama_cpp as _lc

print("torch CUDA :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))
    print("VRAM       :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("llama CUDA :", _lc.llama_supports_gpu_offload())

assert torch.cuda.is_available(), "Включите GPU: Settings → Accelerator → GPU T4 x2"
assert _lc.llama_supports_gpu_offload(), "llama.cpp без CUDA: перезапустите ячейку 2"


In [ ]:
# ── 4. Qwen 2.5 14B Instruct, Q4_K_M, 3 шарда ────────────────────────────
import agent_core as ac

print("repo :", ac.MODEL_REPO)
print("stem :", ac.MODEL_STEM)
print("dir  :", ac.MODELS_DIR)
print("⏳ download/load: ~9 GB, 5–12 min ...")

ac.OFFLINE = False
brain = ac.load_brain(n_gpu_layers=-1, n_ctx=16384, verbose=True)
if ac.is_error(brain):
    raise SystemExit(
        f"Модель не загрузилась: {brain.get('error')}: {brain.get('message')}\n"
        f"hint: {brain.get('hint')}"
    )
print("✅ brain loaded:", type(brain).__name__)


In [ ]:
# ── 5. Preflight бюджета и модели ───────────────────────────────────────
import agent_core as ac

assert not isinstance(brain, ac.ScriptedBrain), "Нужен реальный 14B, не ScriptedBrain"
assert ac.OFFLINE is False
assert len(ac.SYSTEM_PROMPT) <= 3200
budget = ac.prompt_budget(ac.SYSTEM_PROMPT, ac.TOOL_SCHEMAS, ac.N_CTX)
assert budget["est_tokens"] <= 8192, budget
print("budget OK:", budget)
print("brain    :", type(brain).__name__)


In [ ]:
# ── 6. LIVE: одна matrix-подзадача → один batch → export ─────────────────
import json, time
import agent_core as ac

STATIONS = ["IRT", "API", "BSL", "ABG", "BRW", "SHU", "FRD", "CMO", "BOU", "TUC"]
DATES = ["2024-05-14", "2024-05-15", "2024-05-16", "2024-05-17", "2024-05-18"]
PROJECT = "demo_storm_may2024"

PROMPT = (
    "Создай ровно ОДНУ подзадачу create_task_matrix и ничего больше.\n"
    "Не перечисляй ячейки и не добавляй process_matrix_batch: агент сам "
    "выполнит один process_matrix_batch(run_until_complete=True) и экспорт.\n"
    f"events = {json.dumps(DATES, ensure_ascii=False)}\n"
    f"stations = {json.dumps(STATIONS, ensure_ascii=False)}\n"
    'actions = ["fetch_observatory_data", "calculate_derived_components", '
    '"plot_components"]\n'
    f'project_name = "{PROJECT}"\n'
)

print(PROMPT)
STORE = ac.FrameStore()
started = time.time()
result = ac.run_agent_with_planner(
    PROMPT,
    brain=brain,
    store=STORE,
    verbose=True,
    use_llm_planner=True,
)
elapsed = time.time() - started

print("=" * 70)
print("ok       :", result.get("ok"))
print("elapsed  :", round(elapsed, 1), "s")
print("stop     :", result.get("stop_reason"))
print("text     :", str(result.get("text") or "")[:2000])
for i, call in enumerate(result.get("tool_calls", []), 1):
    print(f"{i:02d}. {call.get('tool')} ok={call.get('ok')} error={call.get('error')}")


In [ ]:
# ── 7. Проверка 50 HTML + default/custom ZIP ────────────────────────────
import hashlib, json, zipfile
from pathlib import Path
import project_store as ps

PROJECT = "demo_storm_may2024"
CUSTOM = "storm_report_2024_demo"
FOLDERS = {
    "1": ["irt", "api", "bsl", "abg"],
    "2": ["brw", "shu", "frd"],
    "3": ["cmo", "bou", "tuc"],
}

pdir = Path("projects") / PROJECT
if not pdir.is_dir():
    raise SystemExit(f"Проект не создан: {pdir}")

charts_raw = json.loads((pdir / "charts.json").read_text(encoding="utf-8"))
chart_rows = charts_raw if isinstance(charts_raw, list) else charts_raw.get("charts", [])
html = sorted(pdir.rglob("*.html"))
print("charts.json rows :", len(chart_rows))
print("HTML in project  :", len(html))
print("first/last       :", html[0].name if html else None, "/", html[-1].name if html else None)
if len(html) != 50:
    print("⚠️ matrix не дала 50 HTML. Не выдавайте это за успешный live-прогон.")
    print("   Детерминированный fallback: !python demo_make_50.py")
else:
    print("✅ 50/50 HTML")

# Custom export must not mutate the project.
tree_before = ps.project_tree(PROJECT)
charts_hash = hashlib.sha256((pdir / "charts.json").read_bytes()).hexdigest()
manifest_before = json.loads((pdir / "manifest.json").read_text(encoding="utf-8"))
created_before = manifest_before.get("created_at")

default_zip = ps.export_project(
    PROJECT,
    archive_name=PROJECT,
    sort_by=["station", "start_date", "components", "source", "file"],
)
custom_zip = ps.export_project(
    PROJECT,
    archive_name=CUSTOM,
    folders=FOLDERS,
    sort_by=["date", "station"],
    source_order=["OMNI", "INTERMAGNET", "KYOTO", "SUPERMAG", "NOAA", "ISWA"],
)

manifest_after = json.loads((pdir / "manifest.json").read_text(encoding="utf-8"))
print("created_at stable:", manifest_after.get("created_at") == created_before)
print("tree stable      :", ps.project_tree(PROJECT) == tree_before)
print("charts.json same :", hashlib.sha256((pdir / "charts.json").read_bytes()).hexdigest() == charts_hash)

for label, payload in (("default", default_zip), ("custom", custom_zip)):
    print("\n", label, json.dumps(payload, ensure_ascii=False, default=str)[:1200])
    if payload.get("ok"):
        zp = Path(payload["path"])
        with zipfile.ZipFile(zp) as z:
            names = z.namelist()
            charts = [n for n in names if n.endswith(".html") and not n.endswith("index.html")]
            print("  zip:", zp.name, "bytes", zp.stat().st_size, "entries", len(names), "charts", len(charts))
            print("  zip test:", z.testzip())


In [ ]:
# ?? 7a. OPTIONAL: deterministic peak-storm supplement (no model) ????????
# 20 charts for 10?12 May 2024, outside the 50-chart demo. This is a
# deterministic scientific supplement, not a 14B result; label it as such.
# Takes ~1?2 min and writes stress_runs/peak_storm_2024_supplement/.
!python peak_storm_2024.py --output stress_runs/peak_storm_2024_supplement


In [ ]:
# ── 8. Показать 3 графика в ноутбуке ────────────────────────────────────
from IPython.display import IFrame, display

html = sorted((Path("projects") / "demo_storm_may2024").rglob("*.html"))
for path in html[:3]:
    print(path)
    display(IFrame(src=path.as_posix(), width="100%", height=760))

print("\nЕсли IFrame пустой — откройте файл через Files справа:")
print("  /kaggle/working/geomagnetic-agent/projects/demo_storm_may2024/...")


In [ ]:
# ── 9. Ожидаемый partial/empty: ABG 13 May ──────────────────────────────
import json
import agent_core as ac

store = ac.FrameStore()
for station, day in (("ABG", "2024-05-13"), ("IRT", "2024-05-13")):
    payload, note = ac.HANDLERS["fetch_observatory_data"](
        {
            "station_code": station,
            "start_date": day,
            "end_date": day,
            "data_type": "best-avail",
        },
        store,
    )
    print("\n", station, day)
    print("note:", note)
    if ac.is_error(payload):
        print("error:", payload.get("error"))
        print("message:", payload.get("message"))
        print("attempts:", len(payload.get("attempts", [])))
    else:
        print("rows:", payload.get("rows"), "handle:", payload.get("frame_handle"))


In [ ]:
# ?? 10a. Security probes: bad project/archive names ?????????????????????
import json
import project_store as ps

# '..' is rejected before any directory is created.
bad_project = ps.create_project("..")
print("bad project:", json.dumps(bad_project, ensure_ascii=False, default=str))

# Archive traversal is rejected before any ZIP is written.
bad_archive = ps.export_project("demo_storm_may2024", archive_name="../../escape")
print("bad archive:", json.dumps(bad_archive, ensure_ascii=False, default=str))

assert bad_project.get("ok") is False
assert bad_archive.get("ok") is False
print("security probe: ok")


In [ ]:
# ── 10. LIVE multi-turn: один FrameStore, H → D ─────────────────────────
import agent_core as ac

store = ac.FrameStore()
q1 = (
    "Станция IRT, 2024-05-14. Загрузи данные, вычисли H, D, I и построй H. "
    "Верни точный handle derived-кадра."
)
r1 = ac.run_agent(
    q1, brain=brain, store=store, max_rounds=10, max_tool_calls=12, verbose=True
)
print("TURN1 ok:", r1.get("ok"), "plots:", r1.get("plots"))
for c in r1.get("tool_calls", []):
    print("  ", c.get("tool"), "ok=", c.get("ok"), "args=", c.get("arguments"))

q2 = (
    "Теперь построй D из того же derived-кадра, не скачивай данные заново. "
    "Используй handle из предыдущего хода."
)
r2 = ac.run_agent(
    q2, brain=brain, store=store, max_rounds=8, max_tool_calls=10, verbose=True
)
print("TURN2 ok:", r2.get("ok"), "plots:", r2.get("plots"))
for c in r2.get("tool_calls", []):
    print("  ", c.get("tool"), "ok=", c.get("ok"), "args=", c.get("arguments"))


In [ ]:
# ── 11. OPTIONAL: Gradio UI с публичной ссылкой ─────────────────────────
# Запускайте только если нужен живой чат. Ячейка блокирует ноутбук; остановите
# её после показа. Ссылка появится в выводе (Gradio share).
import gradio_app as ga

demo = ga.build_demo(brain)
demo.launch(share=True, debug=False)


## Что скачать и что показать

После ячейки 7 скачайте:

- `exports/demo_storm_may2024.zip` — canonical tree, 50 HTML;
- `exports/storm_report_2024_demo.zip` — custom `1/2/3`, 50 HTML.

Что показать:

1. 3 HTML: H/D/I на отдельных панелях, единицы `nT`/`deg`.
2. `charts.json` — 50 записей, source/station/date/components.
3. `index.csv` custom ZIP — 50 строк, `archive_path` с `1/2/3`, но `file`
   остаётся каноническим.
4. ABG 13 May — `no_data`, без выдуманных данных.
5. Multi-turn: H, затем D из того же handle.
6. Optional: Gradio share-ссылка.

## Ограничения

- 50 HTML — demo, не 300-gate. Полный 300-gate на новом HEAD не перезапускался.
- Dst/Kp для supplement — Kyoto WDC; Dst может быть provisional.
- NOAA/ISWA current-window не являются историческим маем 2024.
- SuperMAG SME/SML требуют `SUPERMAG_LOGON`; для 50-chart demo не нужны.
